# 05 — Exponential Smoothing Forecasting v2

**Gold Nexus Alpha — professor-style smoothing revision**

This notebook revises the smoothing stage so it matches the same evaluation standard as the Naive and revised ARIMA notebooks.

Main corrections:

1. Keep the locked Dataset A and official validation/test dates.
2. Show the gold price series before modeling.
3. Keep static multi-step smoothing forecasts as a diagnostic.
4. Add one-step rolling smoothing forecasts for fair daily evaluation.
5. Compare Simple Exponential Smoothing, Holt Trend, and Damped Holt.
6. Keep Holt-Winters as a professor-style seasonal diagnostic only if it runs cleanly.
7. Select the final smoothing candidate by validation RMSE.
8. Export JSON artifacts for the frontend and later model comparison notebook.

Important interpretation:

- Static multi-step smoothing can underforecast a large 2024–2026 gold breakout.
- One-step rolling smoothing updates after every actual observation and is more comparable to Naive.

In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the clean Colab → GitHub artifact workflow used across Gold Nexus Alpha.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

print("✅ Repo ready:", PROJECT_DIR)
print("✅ Branch:", BRANCH)
print("✅ UTC time:", datetime.now(timezone.utc).isoformat())

In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load smoothing/time-series libraries.
# - Keep the notebook CPU-safe and Colab-compatible.
# ======================================================================================

import sys
import json
import math
import glob
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import statsmodels
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels

from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("statsmodels:", statsmodels.__version__)

In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Dataset A: data/aligned/model_ready_univariate.csv
# - Fall back to weekday_clean_matrix.csv or uploaded matrix only if needed.
# - Lock the official train / validation / test dates.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

LONG_UNIVARIATE_START = "1968-01-04"
LONG_UNIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "1968-01-04"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

TARGET_COL = "gold_price"

# Rolling smoothing configuration.
# These values are optimized once on train, then updated one observation at a time.
ROLLING_MODEL_NAMES = [
    "SES Rolling One-Step",
    "Holt Rolling One-Step",
    "Damped Holt Rolling One-Step",
]

candidate_inputs = [
    ALIGNED_DIR / "model_ready_univariate.csv",
    ALIGNED_DIR / "weekday_clean_matrix.csv",
    PROJECT_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
]

candidate_inputs += sorted(Path("/content").glob("*model*ready*univariate*.csv"))
candidate_inputs += sorted(Path("/content").glob("*Gold*Matrix*.csv"))
candidate_inputs += sorted(Path("/content").glob("*gold*matrix*.csv"))

INPUT_PATH = None
for path in candidate_inputs:
    if path.exists():
        INPUT_PATH = path
        break

if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find model_ready_univariate.csv, weekday_clean_matrix.csv, or an uploaded Gold_Matrix CSV. "
        "Run Notebook 03 first or upload the current matrix CSV."
    )

print("✅ Input detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")

if TARGET_COL not in raw_df.columns:
    raise ValueError(f"Input file must contain target column '{TARGET_COL}'.")

raw_df["date"] = pd.to_datetime(raw_df["date"], errors="coerce")
raw_df = raw_df.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

raw_df["weekday"] = raw_df["date"].dt.dayofweek
weekend_rows_detected = int(raw_df["weekday"].isin([5, 6]).sum())
if weekend_rows_detected > 0:
    print(f"⚠️ Weekend rows detected in fallback input: {weekend_rows_detected}. Removing Saturday/Sunday rows.")
    raw_df = raw_df[~raw_df["weekday"].isin([5, 6])].copy()
raw_df = raw_df.drop(columns=["weekday"], errors="ignore")

model_df = raw_df[
    (raw_df["date"] >= pd.Timestamp(LONG_UNIVARIATE_START)) &
    (raw_df["date"] <= pd.Timestamp(LONG_UNIVARIATE_END))
][["date", TARGET_COL]].copy()

model_df[TARGET_COL] = pd.to_numeric(model_df[TARGET_COL], errors="coerce")
model_df = model_df.dropna(subset=[TARGET_COL]).sort_values("date").reset_index(drop=True)

train_df = model_df[(model_df["date"] >= TRAIN_START) & (model_df["date"] <= TRAIN_END)].copy()
validation_df = model_df[(model_df["date"] >= VALIDATION_START) & (model_df["date"] <= VALIDATION_END)].copy()
test_df = model_df[(model_df["date"] >= TEST_START) & (model_df["date"] <= TEST_END)].copy()

if train_df.empty or validation_df.empty or test_df.empty:
    raise ValueError("One or more locked split windows are empty. Check Notebook 03 outputs and dates.")

gold_ts = model_df.set_index("date")[TARGET_COL].asfreq("B")
train_ts = train_df.set_index("date")[TARGET_COL].asfreq("B").dropna()
validation_ts = validation_df.set_index("date")[TARGET_COL].asfreq("B").dropna()
test_ts = test_df.set_index("date")[TARGET_COL].asfreq("B").dropna()
train_validation_ts = pd.concat([train_ts, validation_ts]).sort_index()

print("✅ Dataset A ready")
print("Rows:", len(model_df))
print("Train rows:", len(train_ts), train_ts.index.min().date(), "to", train_ts.index.max().date())
print("Validation rows:", len(validation_ts), validation_ts.index.min().date(), "to", validation_ts.index.max().date())
print("Test rows:", len(test_ts), test_ts.index.min().date(), "to", test_ts.index.max().date())
display(model_df.head())
display(model_df.tail())

In [ ]:
# ======================================================================================
# CELL 4 — MAIN EXPONENTIAL SMOOTHING MODELING LOGIC
# ======================================================================================
# Purpose:
# - Professor-style smoothing workflow: plot series → static smoothing → rolling smoothing.
# - Static forecasts are kept as diagnostics.
# - Rolling one-step forecasts are used as the primary fair comparison mode.
# ======================================================================================


def json_safe(value):
    """Convert pandas/numpy objects into JSON-safe Python values."""
    import pandas as pd
    import numpy as np
    import math
    if value is None:
        return None
    if isinstance(value, (pd.Timestamp,)):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    if isinstance(value, (np.ndarray, list, tuple)):
        return [json_safe(x) for x in value]
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    return value

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    clean_payload = json_safe(payload)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean_payload, f, indent=2)
    print(f"✅ Wrote {path}")

def metrics(actual, prediction):
    actual = pd.Series(actual).astype(float)
    prediction = pd.Series(prediction).astype(float)
    aligned = pd.concat([actual.rename("actual"), prediction.rename("prediction")], axis=1).dropna()
    if aligned.empty:
        return {"n": 0, "mae": None, "mse": None, "rmse": None, "mape": None}
    err = aligned["actual"] - aligned["prediction"]
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    denom = aligned["actual"].replace(0, np.nan)
    mape = float((np.abs(err / denom).dropna().mean()) * 100)
    return {"n": int(len(aligned)), "mae": mae, "mse": mse, "rmse": rmse, "mape": mape}

def path_records(df, max_rows=None):
    out = df.copy()
    if max_rows is not None and len(out) > max_rows:
        out = out.tail(max_rows).copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
    return out.replace({np.nan: None}).to_dict(orient="records")

def add_split_lines(ax, validation_start, test_start):
    ax.axvline(pd.Timestamp(validation_start), linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(pd.Timestamp(test_start), linestyle="--", linewidth=1, alpha=0.7)


RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

# 1) Plot the original gold price series first.
fig, ax = plt.subplots(figsize=(14, 5))
gold_ts.dropna().plot(ax=ax)
ax.set_title("Gold Price Time Series — Dataset A")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
add_split_lines(ax, VALIDATION_START, TEST_START)
plt.show()

# 2) Helper functions for smoothing fits.
def fit_static_model(model_name, train_series, forecast_index):
    model_name = str(model_name)
    try:
        if model_name == "SES Static":
            fit = SimpleExpSmoothing(train_series, initialization_method="estimated").fit(optimized=True)
        elif model_name == "Holt Static":
            fit = Holt(train_series, exponential=False, damped_trend=False, initialization_method="estimated").fit(optimized=True)
        elif model_name == "Damped Holt Static":
            fit = Holt(train_series, exponential=False, damped_trend=True, initialization_method="estimated").fit(optimized=True)
        elif model_name == "Holt-Winters Weekday Static":
            # Weekday seasonality is a diagnostic only. It is not forced to win.
            fit = ExponentialSmoothing(
                train_series,
                trend="add",
                seasonal="add",
                seasonal_periods=5,
                initialization_method="estimated"
            ).fit(optimized=True)
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        pred = fit.forecast(len(forecast_index))
        pred.index = forecast_index

        fitted = pd.Series(fit.fittedvalues, index=train_series.index, name="fitted")
        params = {k: v for k, v in fit.params.items() if isinstance(v, (int, float, np.integer, np.floating)) or v is None}

        return {"ok": True, "fit": fit, "prediction": pred, "fitted": fitted, "params": params, "error": None}

    except Exception as e:
        return {"ok": False, "fit": None, "prediction": pd.Series(index=forecast_index, dtype=float), "fitted": pd.Series(dtype=float), "params": {}, "error": str(e)}

def get_last_state_from_fit(fit, train_series, damped=False):
    # Robust extraction of final level/trend state from statsmodels Holt-Winters result.
    level = None
    trend = None
    try:
        level = float(pd.Series(fit.level).dropna().iloc[-1])
    except Exception:
        level = float(train_series.iloc[-1])
    try:
        trend = float(pd.Series(fit.trend).dropna().iloc[-1])
    except Exception:
        recent_diff = train_series.diff().dropna().tail(20)
        trend = float(recent_diff.mean()) if len(recent_diff) else 0.0
    params = fit.params
    alpha = params.get("smoothing_level", None)
    beta = params.get("smoothing_trend", params.get("smoothing_slope", None))
    phi = params.get("damping_trend", params.get("damping_slope", None))
    alpha = 0.2 if alpha is None or pd.isna(alpha) else float(alpha)
    beta = 0.1 if beta is None or pd.isna(beta) else float(beta)
    phi = 0.98 if (damped and (phi is None or pd.isna(phi))) else (1.0 if phi is None or pd.isna(phi) else float(phi))
    return level, trend, alpha, beta, phi

def rolling_ses_forecast(train_series, eval_series, alpha, level_start):
    level = float(level_start)
    rows = []
    for date, actual in eval_series.items():
        pred = level
        rows.append({"date": date, "actual": float(actual), "prediction": float(pred)})
        level = alpha * float(actual) + (1 - alpha) * level
    return pd.DataFrame(rows)

def rolling_holt_forecast(train_series, eval_series, alpha, beta, phi, level_start, trend_start):
    level = float(level_start)
    trend = float(trend_start)
    rows = []
    for date, actual in eval_series.items():
        pred = level + phi * trend
        rows.append({"date": date, "actual": float(actual), "prediction": float(pred)})
        prev_level = level
        level = alpha * float(actual) + (1 - alpha) * (prev_level + phi * trend)
        trend = beta * (level - prev_level) + (1 - beta) * phi * trend
    return pd.DataFrame(rows)

# 3) Static validation models.
static_model_names = [
    "SES Static",
    "Holt Static",
    "Damped Holt Static",
    "Holt-Winters Weekday Static",
]

static_validation_rows = []
static_fits_train = {}

for model_name in static_model_names:
    result = fit_static_model(model_name, train_ts, validation_ts.index)
    static_fits_train[model_name] = result
    if result["ok"]:
        m = metrics(validation_ts, result["prediction"])
        static_validation_rows.append({
            "model_name": model_name,
            "forecast_mode": "static_multi_step_validation",
            **m,
            "parameters": result["params"],
            "fit_error": None,
        })
    else:
        static_validation_rows.append({
            "model_name": model_name,
            "forecast_mode": "static_multi_step_validation",
            "n": 0,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "parameters": {},
            "fit_error": result["error"],
        })

static_validation_table = pd.DataFrame(static_validation_rows).sort_values("rmse", na_position="last").reset_index(drop=True)
print("\nStatic multi-step validation leaderboard")
display(static_validation_table)

# 4) Rolling one-step smoothing validation.
# Use optimized parameters from train fits, then update each day with the actual observation.
ses_fit = static_fits_train["SES Static"]["fit"]
holt_fit = static_fits_train["Holt Static"]["fit"]
damped_fit = static_fits_train["Damped Holt Static"]["fit"]

rolling_validation_paths = {}
rolling_validation_rows = []

if ses_fit is not None:
    alpha = ses_fit.params.get("smoothing_level", 0.2)
    alpha = 0.2 if alpha is None or pd.isna(alpha) else float(alpha)
    try:
        start_level = float(pd.Series(ses_fit.level).dropna().iloc[-1])
    except Exception:
        start_level = float(train_ts.iloc[-1])
    path = rolling_ses_forecast(train_ts, validation_ts, alpha, start_level)
    rolling_validation_paths["SES Rolling One-Step"] = path
    rolling_validation_rows.append({"model_name": "SES Rolling One-Step", "forecast_mode": "one_step_rolling_validation", **metrics(path["actual"], path["prediction"]), "parameters": {"alpha": alpha}})

if holt_fit is not None:
    level, trend, alpha, beta, phi = get_last_state_from_fit(holt_fit, train_ts, damped=False)
    path = rolling_holt_forecast(train_ts, validation_ts, alpha, beta, 1.0, level, trend)
    rolling_validation_paths["Holt Rolling One-Step"] = path
    rolling_validation_rows.append({"model_name": "Holt Rolling One-Step", "forecast_mode": "one_step_rolling_validation", **metrics(path["actual"], path["prediction"]), "parameters": {"alpha": alpha, "beta": beta, "phi": 1.0}})

if damped_fit is not None:
    level, trend, alpha, beta, phi = get_last_state_from_fit(damped_fit, train_ts, damped=True)
    path = rolling_holt_forecast(train_ts, validation_ts, alpha, beta, phi, level, trend)
    rolling_validation_paths["Damped Holt Rolling One-Step"] = path
    rolling_validation_rows.append({"model_name": "Damped Holt Rolling One-Step", "forecast_mode": "one_step_rolling_validation", **metrics(path["actual"], path["prediction"]), "parameters": {"alpha": alpha, "beta": beta, "phi": phi}})

rolling_validation_table = pd.DataFrame(rolling_validation_rows).sort_values("rmse", na_position="last").reset_index(drop=True)
print("\nRolling one-step validation leaderboard")
display(rolling_validation_table)

if rolling_validation_table.empty:
    raise RuntimeError("No rolling smoothing models were successfully evaluated.")

selected_row = rolling_validation_table.iloc[0].to_dict()
SELECTED_MODEL_NAME = selected_row["model_name"]
print("\n✅ Selected smoothing model by rolling validation RMSE:", SELECTED_MODEL_NAME)
display(pd.DataFrame([selected_row]))

# 5) Test evaluation: refit/reinitialize on train + validation, then roll through test.
static_selected_base = SELECTED_MODEL_NAME.replace(" Rolling One-Step", " Static")
train_validation_static_fit = fit_static_model(static_selected_base, train_validation_ts, test_ts.index)

# Static test diagnostic for selected family.
static_test_prediction = train_validation_static_fit["prediction"] if train_validation_static_fit["ok"] else pd.Series(index=test_ts.index, dtype=float)
static_test_metrics = metrics(test_ts, static_test_prediction)

# Rolling test using train+validation state.
if SELECTED_MODEL_NAME == "SES Rolling One-Step":
    base_fit = train_validation_static_fit["fit"]
    alpha = base_fit.params.get("smoothing_level", 0.2)
    alpha = 0.2 if alpha is None or pd.isna(alpha) else float(alpha)
    try:
        start_level = float(pd.Series(base_fit.level).dropna().iloc[-1])
    except Exception:
        start_level = float(train_validation_ts.iloc[-1])
    selected_test_path = rolling_ses_forecast(train_validation_ts, test_ts, alpha, start_level)
    selected_params = {"alpha": alpha}
elif SELECTED_MODEL_NAME == "Holt Rolling One-Step":
    base_fit = train_validation_static_fit["fit"]
    level, trend, alpha, beta, phi = get_last_state_from_fit(base_fit, train_validation_ts, damped=False)
    selected_test_path = rolling_holt_forecast(train_validation_ts, test_ts, alpha, beta, 1.0, level, trend)
    selected_params = {"alpha": alpha, "beta": beta, "phi": 1.0}
else:
    base_fit = train_validation_static_fit["fit"]
    level, trend, alpha, beta, phi = get_last_state_from_fit(base_fit, train_validation_ts, damped=True)
    selected_test_path = rolling_holt_forecast(train_validation_ts, test_ts, alpha, beta, phi, level, trend)
    selected_params = {"alpha": alpha, "beta": beta, "phi": phi}

rolling_test_metrics = metrics(selected_test_path["actual"], selected_test_path["prediction"])

print("\nSelected smoothing static test diagnostic")
display(pd.DataFrame([{**static_test_metrics, "model_name": static_selected_base, "forecast_mode": "static_multi_step_test"}]))

print("\nSelected smoothing rolling test performance")
display(pd.DataFrame([{**rolling_test_metrics, "model_name": SELECTED_MODEL_NAME, "forecast_mode": "one_step_rolling_test"}]))

# 6) Build combined forecast path for frontend.
forecast_path = pd.DataFrame({
    "date": test_ts.index,
    "actual": test_ts.values,
    "static_forecast": static_test_prediction.reindex(test_ts.index).values,
})
rolling_map = selected_test_path.set_index("date")["prediction"]
forecast_path["rolling_forecast"] = forecast_path["date"].map(rolling_map)
forecast_path["residual"] = forecast_path["actual"] - forecast_path["rolling_forecast"]
forecast_path["abs_error"] = forecast_path["residual"].abs()
forecast_path["forecast_mode_default"] = "one_step_rolling"

# 7) Professor-style charts.
fig, ax = plt.subplots(figsize=(14, 5))
train_ts.tail(750).plot(ax=ax, label="Training Actual")
validation_ts.plot(ax=ax, label="Validation Actual")
for name, path in rolling_validation_paths.items():
    ax.plot(path["date"], path["prediction"], label=name, alpha=0.85)
ax.set_title("Exponential Smoothing — Rolling Validation Forecasts")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(forecast_path["date"], forecast_path["actual"], label="Actual Gold Price")
ax.plot(forecast_path["date"], forecast_path["rolling_forecast"], label=f"{SELECTED_MODEL_NAME} Forecast")
ax.plot(forecast_path["date"], forecast_path["static_forecast"], label="Static Multi-Step Diagnostic", alpha=0.65)
ax.set_title("Exponential Smoothing — Test Actual vs Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(forecast_path["date"], forecast_path["residual"], label="Rolling Residual = Actual - Forecast")
ax.axhline(0, linewidth=1)
ax.set_title("Exponential Smoothing — Rolling Test Residuals")
ax.set_xlabel("Date")
ax.set_ylabel("Residual")
ax.legend()
plt.show()

zoom_df = forecast_path.tail(90)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(zoom_df["date"], zoom_df["actual"], marker="o", linestyle="-", label="Actual Gold Price")
ax.plot(zoom_df["date"], zoom_df["rolling_forecast"], marker="o", linestyle="-", label=f"{SELECTED_MODEL_NAME}")
ax.set_title("Exponential Smoothing — Recent Test Zoom")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
ax.legend()
plt.show()

# Objects used by Cell 5.
smoothing_summary = {
    "notebook": "05_exponential_smoothing_forecast.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "project_identity": "Gold Nexus Alpha professor-style forecasting platform",
    "dataset": {
        "name": "Dataset A — Long Univariate",
        "target": TARGET_COL,
        "start": LONG_UNIVARIATE_START,
        "end": LONG_UNIVARIATE_END,
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": len(train_ts)},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": len(validation_ts)},
        "test": {"start": TEST_START, "end": TEST_END, "rows": len(test_ts)},
    },
    "selection_rule": "Final smoothing candidate selected by rolling one-step validation RMSE.",
    "selected_model": {
        "model_name": SELECTED_MODEL_NAME,
        "parameters": selected_params,
        "validation_metrics": selected_row,
        "test_metrics_rolling": rolling_test_metrics,
        "test_metrics_static_diagnostic": static_test_metrics,
    },
    "static_validation_leaderboard": static_validation_table.to_dict(orient="records"),
    "rolling_validation_leaderboard": rolling_validation_table.to_dict(orient="records"),
    "interpretation_notes": [
        "Static multi-step smoothing forecasts are retained as diagnostics because they may lag sharply during regime breaks.",
        "The default frontend chart should use the one-step rolling forecast because it updates after each observed gold price.",
        "Holt-Winters weekday seasonality is diagnostic only and is not forced unless it improves validation evidence."
    ],
}

smoothing_forecast_path_df = forecast_path.copy()

In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Notebook 05 JSON artifacts for model pages and validation notebook.
# ======================================================================================

results_path = MODELS_ARTIFACTS_DIR / "exponential_smoothing_results.json"
forecast_path_json = MODELS_ARTIFACTS_DIR / "exponential_smoothing_forecast_path.json"
page_path = PAGES_ARTIFACTS_DIR / "page_exponential_smoothing.json"

write_json(results_path, smoothing_summary)

write_json(forecast_path_json, {
    "model_name": smoothing_summary["selected_model"]["model_name"],
    "forecast_mode_default": "one_step_rolling",
    "static_forecast_is_diagnostic": True,
    "records": path_records(smoothing_forecast_path_df),
})

page_payload = {
    "page_title": "Exponential Smoothing Forecast",
    "page_subtitle": "Professor-style smoothing model with static diagnostics and one-step rolling evaluation.",
    "artifact_type": "model_page",
    "model_family": "Exponential Smoothing",
    "default_chart_mode": "one_step_rolling",
    "dataset_window": smoothing_summary["dataset"],
    "selected_model": smoothing_summary["selected_model"],
    "leaderboards": {
        "static_validation": smoothing_summary["static_validation_leaderboard"],
        "rolling_validation": smoothing_summary["rolling_validation_leaderboard"],
    },
    "charts": [
        {
            "chart_id": "actual_vs_rolling_forecast",
            "title": "Actual Gold Price vs Rolling Smoothing Forecast",
            "source_artifact": "artifacts/models/exponential_smoothing_forecast_path.json",
            "x": "date",
            "y_actual": "actual",
            "y_forecast": "rolling_forecast"
        },
        {
            "chart_id": "static_forecast_diagnostic",
            "title": "Static Multi-Step Forecast Diagnostic",
            "source_artifact": "artifacts/models/exponential_smoothing_forecast_path.json",
            "x": "date",
            "y_actual": "actual",
            "y_forecast": "static_forecast"
        },
        {
            "chart_id": "rolling_residuals",
            "title": "Rolling Forecast Residuals",
            "source_artifact": "artifacts/models/exponential_smoothing_forecast_path.json",
            "x": "date",
            "y": "residual"
        }
    ],
    "limitations": smoothing_summary["interpretation_notes"],
    "json_first_rule": "Frontend should read all model claims, metrics, and selected-model labels from this artifact."
}

write_json(page_path, page_payload)

print("✅ Notebook 05 v2 exports complete")
print(results_path)
print(forecast_path_json)
print(page_path)

In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 05 revised outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/exponential_smoothing_results.json",
    "artifacts/models/exponential_smoothing_forecast_path.json",
    "artifacts/pages/page_exponential_smoothing.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Revise Notebook 05 smoothing rolling artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ GitHub push complete.")
else:
    print("✅ No changes to commit. Artifacts already match repository.")

print("✅ Notebook 05 v2 professor-style smoothing revision complete.")